23_LLM_clasificar_playground.ipynb
--------------

    
Make sure to set your Diffbot API key in a .env file as follows:
    DIFFBOT_API_KEY=your_diffbot_api_key

In [ ]:
import json
import os
import re
import time
from pathlib import Path

from google import genai
from google.genai.errors import ClientError
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # Automatically finds .env file
api_key = os.getenv('DIFFBOT_API_KEY')

from google import genai


## Definitions

In [ ]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import os   

base_dir = Path.cwd()
csv_path = base_dir / ".." / "data" / "Monitoreo noticias with diffbot fields.csv"

df_articles = pd.read_csv(csv_path)
print(f"Number of rows in DataFrame: {len(df_articles)}")

csv_path = base_dir / ".." / "data" / "dataset articulos Mapa Policia - Hoja 1.csv"

df_dataset = pd.read_csv(csv_path)
print(f"Number of rows in DataFrame: {len(df_dataset)}")


load_dotenv()  # Automatically finds .env file
api_key = os.getenv('GEMINI_API_KEY')

if not api_key:
    raise RuntimeError("Falta GEMINI_API_KEY en el entorno o en .env")



In [ ]:
def extract_json_object(text: str) -> dict:
    """Extract the first JSON object from a model response text."""
    if not text:
        return {}

    cleaned = text.strip()

    # Handle Markdown code fences like ```json ... ```.
    fenced_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", cleaned, flags=re.DOTALL)
    if fenced_match:
        cleaned = fenced_match.group(1)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Fallback: find the first balanced {...} block.
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = cleaned[start:end + 1]
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            return {}

    return {}


def generate_with_retry(prompt: str, model_name: str) -> tuple[str, str]:
    """Return (response_text, error_message). Retries on 429 and never raises."""
    backoff = INITIAL_BACKOFF_SECONDS

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
            )
            return (response.text or "", "")
        except ClientError as exc:
            error_text = str(exc)
            is_rate_limit = "429" in error_text or "RESOURCE_EXHAUSTED" in error_text

            if is_rate_limit and attempt < MAX_RETRIES:
                print(
                    f"Rate limit (429) on attempt {attempt}/{MAX_RETRIES}. "
                    f"Retrying in {backoff}s..."
                )
                time.sleep(backoff)
                backoff *= 2
                continue

            return ("", error_text)
        except Exception as exc:  # Failsafe for unexpected SDK/network errors.
            return ("", str(exc))

    return ("", "Unknown error after retries")



In [ ]:
df_articles.sample()

## Prompts

In [ ]:
# df_articles = df_articles[179:182]

MODEL_NAME = "gemini-3.5-flash-lite"

n_i = 11
n_f = 14

question = "Responde 'SI', 'NO', o 'NO SE PUEDE DETERMINAR' si el hecho ocurrió en CABA (Ciudad de Buenos Aires). Además, indica la ubicación, la fuerza de seguridad involucrada y la fecha del hecho si es posible. Con esta información, genera un JSON con las claves: 'ID', 'ocurrio_en_CABA', 'ubicacion', 'fuerza_de_seguridad', 'fecha_del_hecho'. Si no se puede determinar alguna de estas claves, asigna el valor 'NO SE PUEDE DETERMINAR'."

client = genai.Client(api_key=api_key)

MAX_RETRIES = 5
INITIAL_BACKOFF_SECONDS = 20


In [ ]:

results = []
for index, row in df_articles.iloc[n_i:n_f].iterrows():
    article_id = str(row["ID"]).strip()
    html_code = str(row["diffbot_html"]).strip()
    if not html_code or html_code == "nan":
        print(f"Row {index} (ID: {article_id}) has no HTML code. Skipping.")
        continue

    prompt = f"Responde en español. {question}\n\nID del artículo: {article_id}\nHTML: {html_code}"
    response_text, error_message = generate_with_retry(prompt, MODEL_NAME)

    if error_message:
        print(f"\n[{index}] Article ID: {article_id}")
        print("-" * 80)
        print(f"Error calling model: {error_message}")
        print("-" * 80)
        results.append(
            {
                "ID": article_id,
                "ocurrio_en_CABA": "NO SE PUEDE DETERMINAR",
                "ubicacion": "NO SE PUEDE DETERMINAR",
                "fuerza_de_seguridad": "NO SE PUEDE DETERMINAR",
                "fecha_del_hecho": "NO SE PUEDE DETERMINAR",
                "raw_response": "",
                "error": error_message,
            }
        )
        continue

    # response_tokens = count_tokens(client, MODEL_NAME, response_text)

    # print(f"\n[{index}] Article ID: {article_id}")
    # print("-" * 80)
    # print(response_text)
    # print("-" * 80)

    parsed = extract_json_object(response_text)
    if parsed:
        parsed.setdefault("ID", article_id)
        results.append(parsed)
    else:
        results.append(
            {
                "ID": article_id,
                "ocurrio_en_CABA": "NO SE PUEDE DETERMINAR",
                "ubicacion": "NO SE PUEDE DETERMINAR",
                "fuerza_de_seguridad": "NO SE PUEDE DETERMINAR",
                "fecha_del_hecho": "NO SE PUEDE DETERMINAR",
                "raw_response": response_text,
            }
        )



In [ ]:
df_results = pd.DataFrame(results)
df_results

In [ ]:
df_dataset

## ¿Es CABA?

In [ ]:
# Merge df_results and df_dataset on the matching ID keys
df_combined = df_results.merge(
    df_dataset,
    left_on='ID',
    right_on='id_art',
    how='inner'  # Use how='left' if you want to keep all rows from df_results
)

# Select and rename columns as requested
df_caba = df_combined[['ID', 'es caba?', 'ocurrio_en_CABA']].rename(
    columns={'es caba?': 'es_CABA_REAL', 'ocurrio_en_CABA': 'es_CABA_Estimado'}
)

df_caba

In [ ]:
base_dir = Path.cwd()
csv_path = base_dir / ".." / "data" / "clasificacion_caba_merged.csv"

df_results = pd.read_csv(csv_path)
print(f"Number of rows in DataFrame: {len(df_results)}")


In [ ]:
# Merge df_results and df_dataset on the matching ID keys
df_combined = df_results.merge(
    df_dataset,
    left_on='ID',
    right_on='id_art',
    how='inner'  # Use how='left' if you want to keep all rows from df_results
)

# Select and rename columns as requested
df_caba = df_combined[['ID', 'es caba?', 'ocurrio_en_CABA']].rename(
    columns={'es caba?': 'es_CABA_REAL', 'ocurrio_en_CABA': 'es_CABA_Estimado'}
)

df_caba

In [ ]:
import pandas as pd

# Basic contingency table (counts with row/column totals)
contingency_table = pd.crosstab(
    df_caba['es_CABA_REAL'], 
    df_caba['es_CABA_Estimado'], 
    margins=True, 
    margins_name='Total'
)

print(contingency_table)

In [ ]:
df_caba.loc[df_caba['es_CABA_REAL'] == 'Si', 'es_CABA_REAL'] = 'SI'
df_caba.loc[df_caba['es_CABA_REAL'] == 'no', 'es_CABA_REAL'] = 'NO'
df_caba.loc[df_caba['es_CABA_REAL'] == 'si', 'es_CABA_REAL'] = 'SI'
df_caba.loc[df_caba['es_CABA_REAL'] == 'No', 'es_CABA_REAL'] = 'NO'

# Format cross-tabulation as percentages with a heat gradient
ctable = pd.crosstab(df_caba['es_CABA_REAL'], df_caba['es_CABA_Estimado'], normalize='index') * 100

(ctable
 .style
 .background_gradient(cmap='Blues')  # Visual color intensity
 .format("{:.1f}%")                  # Round to 1 decimal place
 .set_caption("Real vs. Estimado - CABA Classification")
)

Save to CSV.

In [ ]:

# if results:
#     output_path = base_dir / ".." / "data" / f"clasificacion_caba_{n_i}_{n_f}.csv"
#     pd.DataFrame(results).to_csv(output_path, index=False)
#     print(f"Saved {len(results)} rows to: {output_path}")
# else:
#     print("No rows were processed. CSV file was not created.")
